In [1]:
import argparse
import logging
from espnet2.bin.asr_inference import Speech2Text
import gc
import os
import csv
from speechbrain.inference.ASR import WhisperASR
from speechbrain.inference.ASR import EncoderASR
from utils.read_transcription import *
from utils.normalise_text import *
from pathlib import Path
from hyperpyyaml import load_hyperpyyaml
import librosa
import torch
from utils.meta import get_audio_info
from utils.apply_vad import *
from utils.list_files import list_files
from utils.VAD_chunk import *
from utils.wer_chunk import wer_chunk
from utils.logging_config import setup_logging
from utils.wer_segment import wer_segment
models = ["wav2vec","whisper-VAD-chunk","whisper-large","whisper-medium","whisper-large-VAD-chunk","wav2vec2-VAD-chunk"]
import gc
gc.collect()

Failed to import Flash Attention, using ESPnet default: No module named 'flash_attn'


0

In [2]:
wer_hparams = load_hyperpyyaml("""wer_stats: !new:speechbrain.utils.metric_stats.ErrorRateStats""")


In [3]:
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
w2v = EncoderASR.from_hparams(source="/vol/experiments3/imbenamor/TAPAS-FRAIS/models/asr-wav2vec2-commonvoice-fr", savedir="/vol/experiments3/imbenamor/TAPAS-FRAIS/models/asr-wav2vec2-commonvoice-fr", run_opts={"device":"cuda:1"})
whisper_med = WhisperASR.from_hparams(source="/vol/experiments3/imbenamor/TAPAS-FRAIS/models/asr-whisper-medium-commonvoice-fr",savedir="/vol/experiments3/imbenamor/TAPAS-FRAIS/models/asr-whisper-medium-commonvoice-fr", run_opts={"device":"cuda:1"})
whisper_large = WhisperASR.from_hparams(source="/vol/experiments3/imbenamor/TAPAS-FRAIS/models/asr-whisper-large-v2-commonvoice-fr",savedir="/vol/experiments3/imbenamor/TAPAS-FRAIS/models/asr-whisper-large-v2-commonvoice-fr", run_opts={"device":"cuda:1"})

speech2text_ester = Speech2Text(
            "/home/rouas/experiments/SpeechRecognition/saved_models/espnet2-conformer-FR/asr_conformer_config.yaml",
            "/home/rouas/experiments/SpeechRecognition/saved_models/espnet2-conformer-FR/asr_conformer.pth",
            device="cuda:1"
        )
speech2text = Speech2Text(
            "/vol/experiments3/rouas/SpeechRecognition/saved_models/espnet2-commonvoice-conformer-FR/asr_commonvoice_conformer_FR_config.yaml",
            "/vol/experiments3/rouas/SpeechRecognition/saved_models/espnet2-commonvoice-conformer-FR/asr_commonvoice_conformer_FR.pth",
            device="cuda:1"
        )

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

speechbrain.integrations.huggingface.huggingface - Wav2Vec2Model is frozen.


Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

speechbrain.integrations.huggingface.huggingface - WhisperModel is frozen.
speechbrain.integrations.huggingface.whisper - whisper encoder-decoder is frozen.


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

speechbrain.integrations.huggingface.huggingface - WhisperModel is frozen.
speechbrain.integrations.huggingface.whisper - whisper encoder-decoder is frozen.


In [4]:
wav_data="/vol/corpora/Daoudi/Data/Monologue/PD" 
ref_trans= "/vol/experiments3/imbenamor/TAPAS-FRAIS/data/Transcript_Monologue_final/PD"
log_file = "khalid/khalid_PD_spon"
csv_path = "khalid/khalid_PD_spon.csv"
pred_folder = "khalid"

In [5]:
def list_files(trans_dir):
    tg_to_wav = {}
    for f in sorted(os.listdir(trans_dir)):
        for w in os.listdir(wav_data):
            if f.endswith(".txt"):
                if f.split("-")[1].split(".")[0] in w:
                    tg_to_wav[f] = w

    return tg_to_wav
tg_to_wav = list_files(ref_trans)
tg_to_wav


{'1PD-AGJI.txt': '1PD-AGJI-image.wav',
 '1PD-ASTW.txt': '1PD-ASTW-image.wav',
 '1PD-BGRG.txt': '1PD-BGRG-image.wav',
 '1PD-BHEW.txt': '1PD-BHEW-image.wav',
 '1PD-BSQI.txt': '1PD-BSQI-image.wav',
 '1PD-DAAY.txt': '1PD-DAAY-image.wav',
 '1PD-DKCM.txt': '1PD-DKCM-image.wav',
 '1PD-FDOD.txt': '1PD-FDOD-image.wav',
 '1PD-FSKS.txt': '1PD-FSKS-image.wav',
 '1PD-HVQV.txt': '1PD-HVQV-image.wav',
 '1PD-IXVU.txt': '1PD-IXVU-image.wav',
 '1PD-KTTU.txt': '1PD-KTTU-image.wav',
 '1PD-LDBC.txt': '1PD-LDBC-image.wav',
 '1PD-LEZQ.txt': '1PD-LEZQ-image.wav',
 '1PD-NHVW.txt': '1PD-NHVW-image.wav',
 '1PD-OAAY.txt': '1PD2-OAAY-image.wav',
 '1PD-OFAS.txt': '1PD-OFAS-image.wav',
 '1PD-OZNC.txt': '1PD-OZNC-image.wav',
 '1PD-OZSZ.txt': '1PD-OZSZ-image.wav',
 '1PD-PXMX.txt': '1PD-PXMX-image.wav',
 '1PD-UVLI.txt': '1PD2-UVLI-image.wav',
 '1PD-WVGL.txt': '1PD-WVGL-image.wav',
 '1PD-XFZE.txt': '1PD2-XFZE-image.wav',
 '1PD-YIWI.txt': '1PD-YIWI-image.wav',
 '1PD-ZNPV.txt': '1PD-ZNPV-image.wav',
 '1PD-ZPHY.txt': '1PD-

In [6]:
len(tg_to_wav)

33

In [7]:
import re

def clean_transcription(text):
    # 1. Garder seulement la partie après ***
    if "***" in text:
        text = text.split("***", 1)[1]

    # 2. Supprimer contenu entre [] et ()
    text = re.sub(r"\[.*?\]", "", text)
    text = re.sub(r"\(.*?\)", "", text)

    # 3. Supprimer # mais garder contenu
    text = text.replace("#", "")

    # 4. Nettoyage des espaces multiples
    text = re.sub(r"\s+", " ", text)

    # 5. Nettoyage des espaces en début/fin de lignes
    lines = [line.strip() for line in text.split("\n") if line.strip()]

    return (" ".join(lines)).split(" ")




In [8]:
import os
import csv
import torch
from hyperpyyaml import load_hyperpyyaml
from speechbrain.utils.metric_stats import ErrorRateStats
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
# ===================== WER CONFIG =====================
wer_hparams = load_hyperpyyaml("""
wer_stats: !new:speechbrain.utils.metric_stats.ErrorRateStats
""")

# ===================== WER FUNCTION =====================
def wer_chunk(results, words):
    hyp = ""
    for r in results:
        hyp += r["text"] + " "
    ref = " ".join(words)

    hyp_norm = normalization(hyp)
    ref_norm = normalization(ref)

    wer_hparams["wer_stats"].clear()
    wer_hparams["wer_stats"].append(
        ids=[0],
        predict=[hyp_norm],
        target=[ref_norm]
    )

    stats = wer_hparams["wer_stats"].summarize()

    S = stats["substitutions"]
    D = stats["deletions"]
    I = stats["insertions"]
    WER = stats["WER"]

    print(f'WER={WER:.4f}, S={S}, D={D}, I={I}')

    return ref_norm, hyp_norm, WER, S, D, I


# ===================== CORPUS STATS =====================
corpus_stats = {
    "w2vec": {"S": 0, "D": 0, "I": 0, "N": 0},
    "whisper": {"S": 0, "D": 0, "I": 0, "N": 0},
    "whisper_large": {"S": 0, "D": 0, "I": 0, "N": 0},
    "conf_cv": {"S": 0, "D": 0, "I": 0, "N": 0},
    "conf_ester": {"S": 0, "D": 0, "I": 0, "N": 0},
    "hmm": {"S": 0, "D": 0, "I": 0, "N": 0},
}

# ===================== CSV =====================
with open(csv_path, "w", newline="", encoding="utf-8") as f:

    fieldnames = [
        "filename", "duration_sec", "samplerate", "channels",

        "trans_w2vec_vad_chunk", "WER_w2vec_vad_chunk", "S_w2vec", "D_w2vec", "I_w2vec",
        "trans_whisper_vad_chunk", "WER_whisper_vad_chunk", "S_whisper", "D_whisper", "I_whisper",
        "trans_whisper_large_vad_chunk", "WER_whisper_large_vad_chunk", "S_whisper_large", "D_whisper_large", "I_whisper_large",
        "trans_conf_cv_vad_chunk", "WER_conf_cv_vad_chunk", "S_conf_cv", "D_conf_cv", "I_conf_cv",
        "trans_conf_ester_vad_chunk", "WER_conf_ester_vad_chunk", "S_conf_ester", "D_conf_ester", "I_conf_ester",
        "trans_hmm_tdnn_vad_chunk", "WER_hmm_tdnn_vad_chunk", "S_hmm", "D_hmm", "I_hmm"
    ]

    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()

    # ===================== LOOP =====================
    for tg, wav in tg_to_wav.items():

        wav_file = os.path.join(wav_data, wav)
        trans_file = os.path.join(ref_trans, tg)

        if not (os.path.exists(wav_file) and os.path.exists(trans_file)):
            continue

        info = get_audio_info(wav_file)
        row = {"filename": wav, **info}

        # Load audio
        audio_np, sr = read_audio_16k(wav_file)
        wav_tensor = torch.from_numpy(audio_np)

        # VAD
        chunks = vad_chunk_with_timestamps(wav_tensor)

        # Reference
        with open(trans_file, "r", encoding="utf-8") as tf:
            text = tf.read()
        words = clean_transcription(text)

        ref_len = len(normalization(" ".join(words)))

        # ======== W2VEC ========
        results = whisper_transcribe_chunks(w2v, "wav2vec2-VAD-chunk", wav_tensor, chunks)
        _, pred, wer, S, D, I = wer_chunk(results, words)

        row["trans_w2vec_vad_chunk"] = pred
        row["WER_w2vec_vad_chunk"] = wer
        row["S_w2vec"], row["D_w2vec"], row["I_w2vec"] = S, D, I

        corpus_stats["w2vec"]["S"] += S
        corpus_stats["w2vec"]["D"] += D
        corpus_stats["w2vec"]["I"] += I
        corpus_stats["w2vec"]["N"] += ref_len

        # ======== WHISPER MED ========
        results = whisper_transcribe_chunks(whisper_med, "whisper-VAD-chunk", wav_tensor, chunks)
        _, pred, wer, S, D, I = wer_chunk(results, words)

        row["trans_whisper_vad_chunk"] = pred
        row["WER_whisper_vad_chunk"] = wer
        row["S_whisper"], row["D_whisper"], row["I_whisper"] = S, D, I

        corpus_stats["whisper"]["S"] += S
        corpus_stats["whisper"]["D"] += D
        corpus_stats["whisper"]["I"] += I
        corpus_stats["whisper"]["N"] += ref_len

        # ======== WHISPER LARGE ========
        results = whisper_transcribe_chunks(whisper_large, "whisper-large-VAD-chunk", wav_tensor, chunks)
        _, pred, wer, S, D, I = wer_chunk(results, words)

        row["trans_whisper_large_vad_chunk"] = pred
        row["WER_whisper_large_vad_chunk"] = wer
        row["S_whisper_large"], row["D_whisper_large"], row["I_whisper_large"] = S, D, I

        corpus_stats["whisper_large"]["S"] += S
        corpus_stats["whisper_large"]["D"] += D
        corpus_stats["whisper_large"]["I"] += I
        corpus_stats["whisper_large"]["N"] += ref_len

        # ======== CONFORMER CV ========
        
        results = espnet_transcribe_chunks(speech2text, wav_tensor, chunks, sr=16000)

        _, pred, wer, S, D, I = wer_chunk(results, words)

        row["trans_conf_cv_vad_chunk"] = pred
        row["WER_conf_cv_vad_chunk"] = wer
        row["S_conf_cv"], row["D_conf_cv"], row["I_conf_cv"] = S, D, I

        corpus_stats["conf_cv"]["S"] += S
        corpus_stats["conf_cv"]["D"] += D
        corpus_stats["conf_cv"]["I"] += I
        corpus_stats["conf_cv"]["N"] += ref_len

        # ======== CONFORMER ESTER ========
        
        results = espnet_transcribe_chunks(speech2text_ester, wav_tensor, chunks, sr=16000)

        _, pred, wer, S, D, I = wer_chunk(results, words)

        row["trans_conf_ester_vad_chunk"] = pred
        row["WER_conf_ester_vad_chunk"] = wer
        row["S_conf_ester"], row["D_conf_ester"], row["I_conf_ester"] = S, D, I

        corpus_stats["conf_ester"]["S"] += S
        corpus_stats["conf_ester"]["D"] += D
        corpus_stats["conf_ester"]["I"] += I
        corpus_stats["conf_ester"]["N"] += ref_len

        # ======== HMM-TDNN ========
        results = hmmtdnn_transcribe_chunks(
            "/vol/experiments3/imbenamor/TAPAS-FRAIS/src/asr_FR_kaldi_hmm_tdnn.sh",
            wav_tensor, chunks,
            "/vol/experiments3/imbenamor/TAPAS-FRAIS/logs/transcription/rhap",
            sr=16000
        )

        _, pred, wer, S, D, I = wer_chunk(results, words)

        row["trans_hmm_tdnn_vad_chunk"] = pred
        row["WER_hmm_tdnn_vad_chunk"] = wer
        row["S_hmm"], row["D_hmm"], row["I_hmm"] = S, D, I

        corpus_stats["hmm"]["S"] += S
        corpus_stats["hmm"]["D"] += D
        corpus_stats["hmm"]["I"] += I
        corpus_stats["hmm"]["N"] += ref_len

        writer.writerow(row)


# ===================== FINAL CORPUS RESULTS =====================
print("\n===== FINAL CORPUS WER =====")

for model, stats in corpus_stats.items():
    S, D, I, N = stats["S"], stats["D"], stats["I"], stats["N"]

    wer = (S + D + I) / N if N > 0 else 0

    print(f"{model}: WER={wer:.4f} | S={S}, D={D}, I={I}, N={N}")

WER=8.8235, S=2, D=4, I=0


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


WER=8.8235, S=2, D=4, I=0
WER=13.2353, S=4, D=5, I=0
WER=5.8824, S=2, D=2, I=0
WER=10.2941, S=6, D=1, I=0
WER=16.1765, S=9, D=1, I=1
WER=14.6789, S=10, D=6, I=0
WER=21.1009, S=10, D=13, I=0
WER=15.5963, S=9, D=8, I=0
WER=20.1835, S=11, D=7, I=4
WER=17.4312, S=12, D=6, I=1
WER=21.1009, S=17, D=4, I=2
WER=10.6667, S=1, D=7, I=0
WER=14.6667, S=3, D=8, I=0
WER=16.0000, S=3, D=9, I=0
WER=17.3333, S=3, D=10, I=0
WER=13.3333, S=5, D=2, I=3
WER=18.6667, S=4, D=8, I=2
WER=22.1311, S=7, D=20, I=0
WER=26.2295, S=3, D=29, I=0
WER=22.9508, S=6, D=22, I=0
WER=22.1311, S=8, D=19, I=0
WER=13.9344, S=9, D=6, I=2
WER=16.3934, S=8, D=11, I=1
WER=11.3861, S=6, D=16, I=1
WER=48.0198, S=4, D=91, I=2
WER=32.6733, S=7, D=55, I=4
WER=30.1980, S=34, D=17, I=10
WER=13.3663, S=17, D=5, I=5
WER=17.3267, S=19, D=10, I=6
WER=11.9266, S=12, D=11, I=3
WER=47.7064, S=12, D=88, I=4
WER=44.0367, S=14, D=82, I=0
WER=35.7798, S=49, D=14, I=15
WER=15.5963, S=18, D=2, I=14
WER=15.5963, S=19, D=10, I=5
WER=24.4604, S=37, D=30